# Titanic Kaggle — feature-focused, reproducible solution

This notebook is designed to be a strong, clean next iteration for the Kaggle Titanic competition. It focuses on:

1. **Finding useful features** through ablation and interpretable diagnostics.
2. **Comparing a few sensible models** without over-indexing on model choice.
3. **Tuning the selected final model** with a compact, targeted grid search.
4. **Generating `submission.csv`** in the correct Kaggle format.

Expected result from the analysis pass: a feature-engineered linear model with title, family, cabin/ticket, and interaction features is usually competitive with or better than tree ensembles on this small dataset. In my analysis pass, the best local CV result was approximately **0.84 mean stratified CV accuracy**, with the biggest feature gains coming from **normalized titles** and **interaction features** such as `Sex × Pclass` and `Title × Pclass`.

> Note: Kaggle leaderboard score can differ from local CV because the public leaderboard is a small subset of the hidden test labels. Treat local stratified CV as the main guide.

In [1]:
# Core imports
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

In [3]:
# Load data. This works both locally and in Kaggle notebooks.
def find_file(filename):
    candidates = [
        filename,
        f"./data/{filename}",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Could not find {filename}. Put it in the notebook directory or update the path.")

train_path = find_file("train.csv")
test_path = find_file("test.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(train.shape, test.shape)
display(train.head())

(891, 12) (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Quick EDA

The target is `Survived`. Before modeling, inspect missingness and the strongest raw survival patterns. The usual strongest signals are `Sex`, `Pclass`, `Age`, `Fare`, family structure, and cabin/ticket-derived information.

In [5]:
print("Train missing values:")
display(train.isna().sum().sort_values(ascending=False))

print("Test missing values:")
display(test.isna().sum().sort_values(ascending=False))

print("Overall survival rate:", train["Survived"].mean())

for col in ["Sex", "Pclass", "Embarked", "SibSp", "Parch"]:
    print(f"Survival by {col}")
    display(train.groupby(col)["Survived"].agg(["count", "mean"]).sort_values("mean", ascending=False))

Train missing values:


Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
Ticket           0
Fare             0
dtype: int64

Test missing values:


Cabin          327
Age             86
Fare             1
PassengerId      0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
Ticket           0
Embarked         0
dtype: int64

Overall survival rate: 0.3838383838383838
Survival by Sex


...

,count,mean
Embarked,,
C,168,0.553571
Q,77,0.389610
S,644,0.336957


Survival by SibSp


,count,mean
SibSp,,
1,209,0.535885
2,28,0.464286
0,608,0.345395
3,16,0.250000
4,18,0.166667
5,5,0.000000
8,7,0.000000


Survival by Parch


,count,mean
Parch,,
3,5,0.600000
1,118,0.550847
2,80,0.500000
0,678,0.343658
5,5,0.200000
4,4,0.000000
6,1,0.000000


## 2. Leakage-aware feature engineering

The custom transformer below is intentionally placed **inside** the sklearn pipeline. This means its learned mappings, such as age medians and ticket counts, are fit separately inside each CV fold rather than being computed once on the full training set.

Engineered features include:

- `Title`, normalized from `Name`
- `FamilySize`, `FamilyType`, `IsAlone`
- `CabinKnown`, `Deck`
- `TicketPrefix`, `TicketGroupSize`, `TicketPrefixCount`
- `FarePerPerson`, `LogFare`, `LogFarePerPerson`
- `AgeMissing`, `FareMissing`
- fixed `AgeBin`, `FareBin`
- interactions: `SexPclass`, `TitlePclass`, `AgeClass`, `EmbarkedPclass`

In [6]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def _normalize_title(self, title):
        if pd.isna(title):
            return "Unknown"
        title = str(title)
        title = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}.get(title, title)
        rare_titles = {
            "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev",
            "Sir", "Jonkheer", "Dona"
        }
        return "Rare" if title in rare_titles else title

    def _ticket_prefix(self, ticket):
        if pd.isna(ticket):
            return "UNKNOWN"
        s = str(ticket).upper().replace(".", "").replace("/", "").strip()
        parts = s.split()
        if len(parts) == 1 and parts[0].isdigit():
            return "NUMBER"
        prefix = "".join([p for p in parts[:-1] if not p.isdigit()])
        if prefix == "":
            prefix = re.sub(r"\d+", "", parts[0])
        prefix = prefix.strip()
        return prefix if prefix else "NUMBER"

    def _base_features(self, X):
        df = X.copy()
        df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False).map(self._normalize_title)
        df["Surname"] = df["Name"].str.split(",", n=1).str[0].fillna("Unknown")
        df["NameLength"] = df["Name"].fillna("").str.len()
        df["TicketPrefix"] = df["Ticket"].map(self._ticket_prefix)

        df["FamilySize"] = df["SibSp"].fillna(0) + df["Parch"].fillna(0) + 1
        df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
        df["FamilyType"] = pd.cut(
            df["FamilySize"],
            bins=[0, 1, 4, 20],
            labels=["Alone", "Small", "Large"]
        ).astype(str)

        df["CabinKnown"] = df["Cabin"].notna().astype(int)
        df["Deck"] = df["Cabin"].astype(str).str[0]
        df.loc[df["Cabin"].isna(), "Deck"] = "U"
        return df

    def fit(self, X, y=None):
        df = self._base_features(X)
        self.age_group_median_ = df.groupby(["Title", "Sex", "Pclass"])["Age"].median()
        self.age_title_median_ = df.groupby("Title")["Age"].median()
        self.age_global_median_ = df["Age"].median()
        self.ticket_counts_ = df["Ticket"].value_counts(dropna=False).to_dict()
        self.ticket_prefix_counts_ = df["TicketPrefix"].value_counts(dropna=False).to_dict()
        self.fare_median_ = df["Fare"].median()
        self.embarked_mode_ = df["Embarked"].mode().iloc[0] if df["Embarked"].notna().any() else "S"
        return self

    def transform(self, X):
        df = self._base_features(X)
        df["AgeMissing"] = df["Age"].isna().astype(int)
        df["FareMissing"] = df["Fare"].isna().astype(int)

        def fill_age(row):
            if pd.notna(row["Age"]):
                return row["Age"]
            key = (row["Title"], row["Sex"], row["Pclass"])
            if key in self.age_group_median_.index:
                value = self.age_group_median_.loc[key]
                if pd.notna(value):
                    return value
            if row["Title"] in self.age_title_median_.index:
                value = self.age_title_median_.loc[row["Title"]]
                if pd.notna(value):
                    return value
            return self.age_global_median_

        df["Age"] = df.apply(fill_age, axis=1)
        df["Embarked"] = df["Embarked"].fillna(self.embarked_mode_)

        df["TicketGroupSize"] = df["Ticket"].map(self.ticket_counts_).fillna(1).astype(float)
        df["TicketPrefixCount"] = df["TicketPrefix"].map(self.ticket_prefix_counts_).fillna(1).astype(float)

        fare = df["Fare"].fillna(self.fare_median_)
        df["FarePerPerson"] = fare / df["TicketGroupSize"].replace(0, 1)
        df["LogFare"] = np.log1p(fare)
        df["LogFarePerPerson"] = np.log1p(df["FarePerPerson"])

        df["AgeBin"] = pd.cut(
            df["Age"], bins=[0, 12, 18, 35, 60, 100],
            labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"],
            include_lowest=True
        ).astype(str)
        df["FareBin"] = pd.cut(
            fare, bins=[-1, 8, 15, 31, 1000],
            labels=["VeryLow", "Low", "Medium", "High"]
        ).astype(str)

        df["SexPclass"] = df["Sex"].astype(str) + "_P" + df["Pclass"].astype(str)
        df["TitlePclass"] = df["Title"].astype(str) + "_P" + df["Pclass"].astype(str)
        df["AgeClass"] = df["AgeBin"].astype(str) + "_P" + df["Pclass"].astype(str)
        df["EmbarkedPclass"] = df["Embarked"].astype(str) + "_P" + df["Pclass"].astype(str)
        return df

In [7]:
# Compatibility helper for older/newer sklearn versions.
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

feature_sets = {
    "base": {
        "num": ["Pclass", "Age", "SibSp", "Parch", "Fare"],
        "cat": ["Sex", "Embarked"]
    },
    "base_plus_title": {
        "num": ["Pclass", "Age", "SibSp", "Parch", "Fare", "AgeMissing", "NameLength"],
        "cat": ["Sex", "Embarked", "Title"]
    },
    "title_family": {
        "num": ["Pclass", "Age", "SibSp", "Parch", "Fare", "AgeMissing", "NameLength", "FamilySize", "IsAlone"],
        "cat": ["Sex", "Embarked", "Title", "FamilyType"]
    },
    "title_family_cabin_ticket": {
        "num": [
            "Pclass", "Age", "SibSp", "Parch", "Fare", "AgeMissing", "FareMissing",
            "NameLength", "FamilySize", "IsAlone", "CabinKnown", "TicketGroupSize",
            "TicketPrefixCount", "FarePerPerson", "LogFare", "LogFarePerPerson"
        ],
        "cat": ["Sex", "Embarked", "Title", "FamilyType", "Deck", "TicketPrefix", "AgeBin", "FareBin"]
    },
    "all_interactions": {
        "num": [
            "Pclass", "Age", "SibSp", "Parch", "Fare", "AgeMissing", "FareMissing",
            "NameLength", "FamilySize", "IsAlone", "CabinKnown", "TicketGroupSize",
            "TicketPrefixCount", "FarePerPerson", "LogFare", "LogFarePerPerson"
        ],
        "cat": [
            "Sex", "Embarked", "Title", "FamilyType", "Deck", "TicketPrefix", "AgeBin", "FareBin",
            "SexPclass", "TitlePclass", "AgeClass", "EmbarkedPclass"
        ]
    }
}

def make_preprocessor(numeric_features, categorical_features, scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(numeric_steps)
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot())
    ])

    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

def make_pipeline(model, feature_key="all_interactions", scale_numeric=True):
    fs = feature_sets[feature_key]
    return Pipeline([
        ("features", TitanicFeatureEngineer()),
        ("preprocess", make_preprocessor(fs["num"], fs["cat"], scale_numeric=scale_numeric)),
        ("model", model)
    ])

X = train.drop(columns=["Survived"])
y = train["Survived"]

## 3. Feature ablation

This is the most important section for your question. It tests progressively richer feature sets using the same model. The goal is to answer: **which feature blocks actually move CV accuracy?**

In the prior analysis pass, the biggest jumps came from:

1. `Title`, because it captures sex, age, marital status, and social status.
2. Family features, especially `FamilySize` and `FamilyType`.
3. Cabin/ticket features, especially `CabinKnown`, `Deck`, `TicketGroupSize`, and `FarePerPerson`.
4. Interaction features, especially `SexPclass` and `TitlePclass`.

In [8]:
def evaluate_cv(name, pipeline, X=X, y=y, cv=CV):
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return {
        "name": name,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std(),
        "fold_scores": scores
    }

ablation_results = []
for feature_key in feature_sets:
    model = LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)
    pipe = make_pipeline(model, feature_key=feature_key, scale_numeric=True)
    result = evaluate_cv(feature_key, pipe)
    ablation_results.append(result)
    print(f"{feature_key:28s} {result['mean_accuracy']:.4f} ± {result['std_accuracy']:.4f}")

ablation_df = pd.DataFrame([
    {"feature_set": r["name"], "mean_accuracy": r["mean_accuracy"], "std_accuracy": r["std_accuracy"]}
    for r in ablation_results
]).sort_values("mean_accuracy", ascending=False)

display(ablation_df)

base                         0.7991 ± 0.0249
base_plus_title              0.8305 ± 0.0246
title_family                 0.8316 ± 0.0227
title_family_cabin_ticket    0.8316 ± 0.0291
all_interactions             0.8271 ± 0.0325


,feature_set,mean_accuracy,std_accuracy
2,title_family,0.831598,0.022749
3,title_family_cabin_ticket,0.831586,0.029110
1,base_plus_title,0.830487,0.024551
4,all_interactions,0.827091,0.032489
0,base,0.799089,0.024941


## 4. Model comparison

My expectation is that features matter more than model choice here. Still, it is useful to compare a few families:

- Logistic Regression: simple, stable, interpretable, strong with good interactions.
- Random Forest / Extra Trees: robust nonlinear baselines.
- Gradient Boosting: often strong on small tabular data but can overfit.

Optional XGBoost/LightGBM/CatBoost can be added, but this notebook avoids making the main solution depend on non-sklearn packages.

In [9]:
model_specs = [
    (
        "LogisticRegression_L2",
        LogisticRegression(max_iter=5000, solver="liblinear", penalty="l2", C=1.0, random_state=RANDOM_STATE),
        True,
    ),
    (
        "LogisticRegression_L1",
        LogisticRegression(max_iter=5000, solver="liblinear", penalty="l1", C=0.9, random_state=RANDOM_STATE),
        True,
    ),
    (
        "RandomForest",
        RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=4,
            max_features=0.5, random_state=RANDOM_STATE, n_jobs=-1
        ),
        False,
    ),
    (
        "ExtraTrees",
        ExtraTreesClassifier(
            n_estimators=300, max_depth=5, min_samples_leaf=4,
            max_features=0.5, random_state=RANDOM_STATE, n_jobs=-1
        ),
        False,
    ),
    (
        "GradientBoosting",
        GradientBoostingClassifier(
            n_estimators=120, learning_rate=0.04, max_depth=3,
            min_samples_leaf=4, random_state=RANDOM_STATE
        ),
        False,
    ),
]

model_results = []
for name, model, scale_numeric in model_specs:
    pipe = make_pipeline(model, feature_key="all_interactions", scale_numeric=scale_numeric)
    result = evaluate_cv(name, pipe)
    model_results.append(result)
    print(f"{name:24s} {result['mean_accuracy']:.4f} ± {result['std_accuracy']:.4f}")

model_df = pd.DataFrame([
    {"model": r["name"], "mean_accuracy": r["mean_accuracy"], "std_accuracy": r["std_accuracy"]}
    for r in model_results
]).sort_values("mean_accuracy", ascending=False)

display(model_df)

LogisticRegression_L2    0.8271 ± 0.0325
LogisticRegression_L1    0.8282 ± 0.0304
RandomForest             0.8361 ± 0.0362
ExtraTrees               0.8327 ± 0.0269
GradientBoosting         0.8271 ± 0.0358


,model,mean_accuracy,std_accuracy
2,RandomForest,0.836067,0.036153
3,ExtraTrees,0.832747,0.026935
1,LogisticRegression_L1,0.828215,0.030418
4,GradientBoosting,0.827116,0.035765
0,LogisticRegression_L2,0.827091,0.032489


## 5. Targeted tuning

Do not run a massive grid on Titanic. The data is small, and it is easy to overfit local CV or the Kaggle public leaderboard.

The first grid tunes Logistic Regression because it is usually strong with these engineered features. The second grid is a compact Random Forest comparison.

In [10]:
# Tuned Logistic Regression
logistic_pipe = make_pipeline(
    LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE),
    feature_key="all_interactions",
    scale_numeric=True,
)

logistic_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.05, 0.1, 0.2, 0.4, 0.7, 0.9, 1.0, 1.3, 1.8, 2.5, 4.0],
    "model__class_weight": [None, "balanced"],
}

logistic_search = GridSearchCV(
    logistic_pipe,
    param_grid=logistic_grid,
    cv=CV,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
)

logistic_search.fit(X, y)
print("Best Logistic CV:", logistic_search.best_score_)
print("Best Logistic params:", logistic_search.best_params_)

Best Logistic CV: 0.8405867665418227
Best Logistic params: {'model__C': 0.1, 'model__class_weight': None, 'model__penalty': 'l2'}


In [12]:
# Compact Random Forest tuning
rf_pipe = make_pipeline(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    feature_key="all_interactions",
    scale_numeric=False,
)

rf_grid = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [3, 4, 5, 6],
    "model__min_samples_leaf": [2, 4, 6],
    "model__max_features": ["sqrt", 0.5],
}

rf_search = GridSearchCV(
    rf_pipe,
    param_grid=rf_grid,
    cv=CV,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
    verbose=2,
)

rf_search.fit(X, y)
print("Best RF CV:", rf_search.best_score_)
print("Best RF params:", rf_search.best_params_)

Fitting 10 folds for each of 72 candidates, totalling 720 fits
[CV] END model__max_depth=3, model__max_features=sqrt, model__min_samples_leaf=2, model__n_estimators=200; total time=   0.3s
[CV] END model__max_depth=3, model__max_features=sqrt, model__min_samples_leaf=2, model__n_estimators=200; total time=   0.3s
[CV] END model__max_depth=3, model__max_features=sqrt, model__min_samples_leaf=2, model__n_estimators=200; total time=   0.3s
[CV] END model__max_depth=3, model__max_features=sqrt, model__min_samples_leaf=2, model__n_estimators=200; total time=   0.3s
[CV] END model__max_depth=3, model__max_features=sqrt, model__min_samples_leaf=2, model__n_estimators=200; total time=   0.3s
[CV] END model__max_depth=3, model__max_features=sqrt, model__min_samples_leaf=2, model__n_estimators=200; total time=   0.3s
[CV] END model__max_depth=3, model__max_features=sqrt, model__min_samples_leaf=2, model__n_estimators=200; total time=   0.4s
[CV] END model__max_depth=3, model__max_features=sqrt, 

/Users/pratikeliasjacob/miniforge3/envs/kaggle/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/pratikeliasjacob/miniforge3/envs/kaggle/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/pratikeliasjacob/miniforge3/envs/kaggle/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/

[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=4, model__n_estimators=600; total time=   1.3s
[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=4, model__n_estimators=600; total time=   1.1s
[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=6, model__n_estimators=200; total time=   0.5s
[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=6, model__n_estimators=200; total time=   0.6s
[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=6, model__n_estimators=400; total time=   0.5s
[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=6, model__n_estimators=400; total time=   0.6s
[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=6, model__n_estimators=400; total time=   0.6s
[CV] END model__max_depth=4, model__max_features=sqrt, model__min_samples_leaf=6, model__n_estimators=400; total time=

In [13]:
# Pick the best tuned candidate by cross-validated accuracy.
candidates = [
    ("Tuned LogisticRegression", logistic_search.best_score_, logistic_search.best_estimator_, logistic_search.best_params_),
    ("Tuned RandomForest", rf_search.best_score_, rf_search.best_estimator_, rf_search.best_params_),
]

best_name, best_score, best_model, best_params = max(candidates, key=lambda x: x[1])

print("Selected model:", best_name)
print("Selected CV accuracy:", best_score)
print("Selected params:", best_params)

Selected model: Tuned LogisticRegression
Selected CV accuracy: 0.8405867665418227
Selected params: {'model__C': 0.1, 'model__class_weight': None, 'model__penalty': 'l2'}


## 6. Error analysis

This shows which groups the model still struggles with. For Titanic, the hardest cases are often groups where major signals conflict, such as first-class men or third-class women.

In [14]:
oof_pred = cross_val_predict(best_model, X, y, cv=CV, n_jobs=-1)

print("OOF accuracy:", accuracy_score(y, oof_pred))
print("Confusion matrix:")
print(confusion_matrix(y, oof_pred))
print("Classification report:")
print(classification_report(y, oof_pred))

analysis_df = TitanicFeatureEngineer().fit(X).transform(X)
analysis_df["Survived"] = y.values
analysis_df["OOF_Pred"] = oof_pred
analysis_df["Correct"] = (analysis_df["Survived"] == analysis_df["OOF_Pred"]).astype(int)

print("Accuracy by Sex and Pclass:")
display(
    analysis_df.groupby(["Sex", "Pclass"])
    .agg(n=("Correct", "size"), survival_rate=("Survived", "mean"), accuracy=("Correct", "mean"))
    .sort_values("accuracy")
)

print("Accuracy by Title:")
display(
    analysis_df.groupby("Title")
    .agg(n=("Correct", "size"), survival_rate=("Survived", "mean"), accuracy=("Correct", "mean"))
    .sort_values("n", ascending=False)
)

OOF accuracy: 0.8406285072951739
Confusion matrix:
[[498  51]
 [ 91 251]]
Classification report:
              precision    recall  f1-score   support

           0       0.85      0.91      0.88       549
           1       0.83      0.73      0.78       342

    accuracy                           0.84       891
   macro avg       0.84      0.82      0.83       891
weighted avg       0.84      0.84      0.84       891

Accuracy by Sex and Pclass:


n  survival_rate  accuracy
Sex    Pclass                              
female 3       144       0.500000  0.673611
male   1       122       0.368852  0.680328
       3       347       0.135447  0.887608
female 2        76       0.921053  0.921053
male   2       108       0.157407  0.925926
female 1        94       0.968085  0.968085

Accuracy by Title:


,n,survival_rate,accuracy
Title,,,
Mr,517,0.156673,0.845261
Miss,185,0.702703,0.816216
Mrs,126,0.793651,0.825397
Master,40,0.575000,0.925000
Rare,23,0.347826,0.869565


## 7. Feature usefulness diagnostics

For Logistic Regression, coefficients are often more informative than tree feature importances because one-hot features expose the actual categories and interactions being used.

In [15]:
def get_feature_names(fitted_pipeline):
    pre = fitted_pipeline.named_steps["preprocess"]
    names = []
    for name, transformer, cols in pre.transformers_:
        if name == "num":
            names.extend(cols)
        elif name == "cat":
            ohe = transformer.named_steps["onehot"]
            names.extend(ohe.get_feature_names_out(cols))
    return np.array(names)

# Fit selected model on all training data for diagnostics.
best_model.fit(X, y)

if hasattr(best_model.named_steps["model"], "coef_"):
    feature_names = get_feature_names(best_model)
    coefs = best_model.named_steps["model"].coef_[0]
    coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
    coef_df["abs_coef"] = coef_df["coef"].abs()
    print("Most positive survival signals:")
    display(coef_df.sort_values("coef", ascending=False).head(25))
    print("Most negative survival signals:")
    display(coef_df.sort_values("coef", ascending=True).head(25))
else:
    print("Selected model does not expose linear coefficients. Using permutation importance instead.")
    perm = permutation_importance(best_model, X, y, scoring="accuracy", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    perm_df = pd.DataFrame({"feature": X.columns, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std})
    display(perm_df.sort_values("importance_mean", ascending=False))

Most positive survival signals:


,feature,coef,abs_coef
16,Sex_female,0.562243,0.562243
79,SexPclass_female_P2,0.475163,0.475163
21,Title_Master,0.406025,0.406025
64,TicketPrefix_STONO,0.325071,0.325071
7,NameLength,0.297348,0.297348
28,FamilyType_Small,0.294015,0.294015
24,Title_Mrs,0.292435,0.292435
33,Deck_E,0.286097,0.286097
88,TitlePclass_Miss_P2,0.274058,0.274058
78,SexPclass_female_P1,0.262603,0.262603


Most negative survival signals:


,feature,coef,abs_coef
23,Title_Mr,-0.732630,0.732630
17,Sex_male,-0.512705,0.512705
91,TitlePclass_Mr_P2,-0.379070,0.379070
1,Age,-0.345519,0.345519
82,SexPclass_male_P2,-0.316348,0.316348
27,FamilyType_Large,-0.311448,0.311448
2,SibSp,-0.310977,0.310977
121,EmbarkedPclass_S_P3,-0.291625,0.291625
0,Pclass,-0.268475,0.268475
8,FamilySize,-0.250616,0.250616


## 8. Optional soft-voting ensemble

Only keep this if it beats the selected tuned model in CV. On Titanic, simple ensembles sometimes help, but they can also add noise.

In [16]:
# Optional: soft-voting ensemble anchored on the tuned logistic model.
# This section is intentionally conservative.
ensemble = VotingClassifier(
    estimators=[
        ("lr", logistic_search.best_estimator_),
        ("rf", rf_search.best_estimator_),
        ("gb", make_pipeline(
            GradientBoostingClassifier(
                n_estimators=120, learning_rate=0.04, max_depth=3,
                min_samples_leaf=4, random_state=RANDOM_STATE
            ),
            feature_key="all_interactions",
            scale_numeric=False,
        )),
    ],
    voting="soft",
    weights=[2, 1, 1],
)

# VotingClassifier with full pipelines can be a bit slower. Uncomment to test.
# ensemble_scores = cross_val_score(ensemble, X, y, cv=CV, scoring="accuracy", n_jobs=-1)
# print("Ensemble CV:", ensemble_scores.mean(), ensemble_scores.std(), ensemble_scores)

## 9. Final training and Kaggle submission

This trains the selected model on all labeled training rows, predicts the Kaggle test set, and writes `submission.csv`.

In [18]:
best_model.fit(X, y)
test_predictions = best_model.predict(test).astype(int)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions,
})

submission.to_csv("submission.csv", index=False)
print(submission.shape)
print("Predicted test survival rate:", submission["Survived"].mean())
display(submission.head())

(418, 2)
Predicted test survival rate: 0.35645933014354064


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## Summary of findings

The most useful features to keep are:

1. **Sex and Pclass**, especially combined as `SexPclass`.
2. **Title**, normalized into common groups plus `Rare`.
3. **TitlePclass**, which captures important social/age/class interactions.
4. **FamilySize / FamilyType**, because small families tend to behave differently from passengers traveling alone or in very large groups.
5. **CabinKnown and Deck**, because missing cabin information is itself informative.
6. **TicketGroupSize and FarePerPerson**, because raw fare mixes individual and grouped tickets.
7. **AgeMissing and group-imputed Age**, because age missingness and age groups both carry signal.

The model comparison is intentionally secondary. In the analysis pass, tuned Logistic Regression was very competitive because the feature set already encoded the key nonlinear interactions. Random Forest and Gradient Boosting are still useful sanity checks, but do not assume a more complex model is better unless it wins in stratified CV.